# 05 — Optimisation du seuil de décision

**Constat :** la régression logistique retenue obtient un ROC-AUC de 0,84, un rappel de 0,79 et une
précision de 0,50 avec le seuil par défaut (0,5). Ce seuil est un choix arbitraire : le déplacer
change l'équilibre entre fausses alertes (précision) et clients à risque non détectés (rappel),
sans réentraîner le modèle.

**Démarche :**

1. Calculer des probabilités « honnêtes » sur l'entraînement par validation croisée (*out-of-fold*).
2. Voir comment précision, rappel et F1 varient selon le seuil.
3. Choisir le seuil selon un critère explicite (F1, rappel minimum ou coût).
4. Vérifier que ce choix est **stable** (bootstrap).
5. Examiner la **calibration** des probabilités.
6. Confirmer **une seule fois** sur le jeu de test.

> Le jeu de test n'est utilisé qu'à l'étape 6. Le seuil est choisi uniquement avec l'entraînement.
> Prérequis : avoir exécuté `python -m src.features.build_features` pour générer les fichiers
> `X_train.csv`, `X_test.csv`, `y_train.csv` et `y_test.csv`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, confusion_matrix,
    precision_recall_curve, roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline

# Permet d'importer le package src depuis le dossier notebooks/
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from src.features.build_features import build_preprocessor

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RANDOM_STATE = 42
SEUIL_ACTUEL = 0.5

# --- Paramètres de décision (hypothèses à ajuster) ---
RAPPEL_MIN = 0.70          # critère "rappel_min" : détecter au moins 70 % des clients qui partent
COUT_FAUX_POSITIF = 1      # coût d'une offre de rétention envoyée à un client qui ne partait pas
COUT_FAUX_NEGATIF = 5      # coût d'un client qui part sans avoir été détecté (en unités du coût ci-dessus)
CRITERE_RETENU = "cout"    # "f1", "rappel_min" ou "cout"
N_BOOTSTRAP = 200

## 1. Données et modèle de référence

In [ ]:
X_train = pd.read_csv(PROCESSED_DIR / "X_train.csv")
X_test = pd.read_csv(PROCESSED_DIR / "X_test.csv")
y_train = pd.read_csv(PROCESSED_DIR / "y_train.csv").squeeze()
y_test = pd.read_csv(PROCESSED_DIR / "y_test.csv").squeeze()
y_tr, y_te = y_train.to_numpy(), y_test.to_numpy()

print(f"Entraînement : {X_train.shape} | Test : {X_test.shape}")
print(f"Taux de churn : entraînement {y_tr.mean():.3f} | test {y_te.mean():.3f}")


def nouveau_pipeline():
    # Même configuration que src/models/train.py
    return Pipeline([
        ("preprocessor", build_preprocessor()),
        ("classifier", LogisticRegression(
            class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE
        )),
    ])

## 2. Probabilités *out-of-fold* sur l'entraînement

Chaque client reçoit la probabilité prédite par un modèle qui **ne l'a pas vu** à l'entraînement.
Choisir un seuil avec des probabilités déjà « apprises » donnerait un résultat trop optimiste.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
proba_oof = cross_val_predict(
    nouveau_pipeline(), X_train, y_train, cv=cv, method="predict_proba"
)[:, 1]

print(f"ROC-AUC (out-of-fold) : {roc_auc_score(y_tr, proba_oof):.4f}")
print(f"Précision moyenne (aire sous la courbe précision-rappel) : {average_precision_score(y_tr, proba_oof):.4f}")

## 3. Métriques selon le seuil

In [ ]:
SEUILS = np.round(np.arange(0.05, 0.96, 0.01), 2)


def metriques_par_seuil(y, proba, seuils=SEUILS):
    y = np.asarray(y)
    seuils = np.asarray(seuils, dtype=float)
    pred = proba[:, None] >= seuils[None, :]
    tp = (pred & (y[:, None] == 1)).sum(axis=0)
    fp = (pred & (y[:, None] == 0)).sum(axis=0)
    fn = ((~pred) & (y[:, None] == 1)).sum(axis=0)
    tn = ((~pred) & (y[:, None] == 0)).sum(axis=0)
    with np.errstate(divide="ignore", invalid="ignore"):
        precision = np.where(tp + fp > 0, tp / (tp + fp), np.nan)
        rappel = tp / (tp + fn)
        f1 = np.where(
            (tp + fp > 0) & (precision + rappel > 0),
            2 * precision * rappel / (precision + rappel), 0.0,
        )
    return pd.DataFrame({
        "seuil": seuils, "precision": precision, "rappel": rappel, "f1": f1,
        "specificite": tn / (tn + fp), "part_marquee": (tp + fp) / len(y),
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
        "cout": fp * COUT_FAUX_POSITIF + fn * COUT_FAUX_NEGATIF,
    }).set_index("seuil")


courbes = metriques_par_seuil(y_tr, proba_oof)

fig, ax = plt.subplots(figsize=(9, 5))
courbes[["precision", "rappel", "f1"]].plot(ax=ax)
ax.axvline(SEUIL_ACTUEL, color="grey", linestyle="--", label="Seuil actuel (0,5)")
ax.set_xlabel("Seuil de décision")
ax.set_ylabel("Valeur")
ax.set_title("Précision, rappel et F1 selon le seuil (out-of-fold)")
ax.legend()
plt.tight_layout()
plt.show()

jalons = np.round(np.arange(0.30, 0.81, 0.05), 2)
display(courbes.loc[courbes.index.isin(jalons), ["precision", "rappel", "f1", "specificite", "part_marquee"]].round(3))

In [ ]:
precision_c, rappel_c, _ = precision_recall_curve(y_tr, proba_oof)
taux_base = y_tr.mean()

ordre = np.argsort(-proba_oof)
captures = np.cumsum(y_tr[ordre]) / y_tr.sum()
part = np.arange(1, len(y_tr) + 1) / len(y_tr)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(rappel_c, precision_c, label="Modèle")
axes[0].axhline(taux_base, color="grey", linestyle=":", label=f"Taux de base ({taux_base:.2f})")
axes[0].scatter(
    courbes.loc[SEUIL_ACTUEL, "rappel"], courbes.loc[SEUIL_ACTUEL, "precision"],
    color="red", zorder=3, label="Seuil 0,5",
)
axes[0].set_xlabel("Rappel")
axes[0].set_ylabel("Précision")
axes[0].set_title("Courbe précision-rappel")
axes[0].legend()

axes[1].plot(part, captures, label="Modèle")
axes[1].plot([0, 1], [0, 1], "--", color="grey", label="Au hasard")
axes[1].set_xlabel("Part des clients contactés (les plus à risque d'abord)")
axes[1].set_ylabel("Part des churners capturés")
axes[1].set_title("Courbe de gain")
axes[1].legend()
plt.tight_layout()
plt.show()

gain = pd.DataFrame({
    "part_contactee": [0.10, 0.20, 0.30, 0.40, 0.50],
    "churners_captures": [captures[int(p * len(y_tr)) - 1] for p in [0.10, 0.20, 0.30, 0.40, 0.50]],
})
display(gain.round(3))

## 4. Choix du seuil

Trois critères, du plus simple au plus utile :

- **F1** : équilibre précision et rappel, sans tenir compte du métier ;
- **rappel minimum** : le plus haut seuil (donc la meilleure précision) qui garde un rappel d'au moins `RAPPEL_MIN` ;
- **coût** : minimise `fausses alertes × COUT_FAUX_POSITIF + clients non détectés × COUT_FAUX_NEGATIF`.

Les coûts sont des **hypothèses** (paramètres en haut du notebook) : le critère « coût » n'est pertinent
que si leur rapport est réaliste. On teste donc plusieurs rapports plus bas.

In [ ]:
def seuil_f1(c):
    return c["f1"].idxmax()


def seuil_rappel_min(c, rappel_min=None):
    rappel_min = RAPPEL_MIN if rappel_min is None else rappel_min
    eligibles = c[c["rappel"] >= rappel_min]
    return eligibles.index.max() if len(eligibles) else np.nan


def seuil_cout(c):
    return c["cout"].idxmin()


criteres = {
    "f1": seuil_f1(courbes),
    "rappel_min": seuil_rappel_min(courbes),
    "cout": seuil_cout(courbes),
}

colonnes = ["precision", "rappel", "f1", "part_marquee", "cout"]
lignes = [{"critere": nom, "seuil": s, **courbes.loc[s, colonnes].to_dict()} for nom, s in criteres.items()]
lignes.append({"critere": "actuel (0,5)", "seuil": SEUIL_ACTUEL, **courbes.loc[SEUIL_ACTUEL, colonnes].to_dict()})
display(pd.DataFrame(lignes).set_index("critere").round(3))

In [ ]:
# Sensibilité du seuil optimal au rapport coût(faux négatif) / coût(faux positif)
lignes = []
for rapport in [1, 2, 3, 5, 10, 20]:
    cout = courbes["fp"] * 1 + courbes["fn"] * rapport
    s = cout.idxmin()
    lignes.append({
        "cout_FN / cout_FP": rapport, "seuil_optimal": s,
        "precision": courbes.loc[s, "precision"], "rappel": courbes.loc[s, "rappel"],
        "part_marquee": courbes.loc[s, "part_marquee"],
    })
display(pd.DataFrame(lignes).set_index("cout_FN / cout_FP").round(3))

## 5. Stabilité du seuil (bootstrap)

On rééchantillonne les clients de l'entraînement et on recalcule le seuil optimal à chaque fois.
Si le seuil varie beaucoup, il existe une zone plate : mieux vaut choisir le **centre** de cette zone
(la médiane) que le pic exact.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
n = len(y_tr)
stabilite = {"f1": [], "rappel_min": [], "cout": []}

for _ in range(N_BOOTSTRAP):
    idx = rng.integers(0, n, n)
    c = metriques_par_seuil(y_tr[idx], proba_oof[idx])
    stabilite["f1"].append(seuil_f1(c))
    stabilite["rappel_min"].append(seuil_rappel_min(c))
    stabilite["cout"].append(seuil_cout(c))

resume_stabilite = pd.DataFrame({
    nom: {
        "mediane": np.nanmedian(v), "ecart_type": np.nanstd(v),
        "p5": np.nanpercentile(v, 5), "p95": np.nanpercentile(v, 95),
    }
    for nom, v in stabilite.items()
}).T
display(resume_stabilite.round(3))

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, (nom, valeurs) in zip(axes, stabilite.items()):
    ax.hist(valeurs, bins=20, color="steelblue")
    ax.set_title(f"Critère : {nom}")
    ax.set_xlabel("Seuil optimal")
plt.tight_layout()
plt.show()

SEUIL_RETENU = round(float(np.nanmedian(stabilite[CRITERE_RETENU])), 2)
print(f"Critère retenu : {CRITERE_RETENU} -> seuil retenu = {SEUIL_RETENU}")

## 6. Calibration des probabilités

Avec `class_weight="balanced"`, les probabilités sont gonflées : un score de 0,72 ne signifie pas que
72 % de ces clients partent. Comme l'application Streamlit affiche ce chiffre en pourcentage, on
compare le modèle actuel à une version **calibrée** (régression sigmoïde).

Pour des probabilités bien calibrées, le seuil optimal théorique est
`coût(FP) / (coût(FP) + coût(FN))`.

In [ ]:
proba_cal_oof = cross_val_predict(
    CalibratedClassifierCV(nouveau_pipeline(), method="sigmoid", cv=5),
    X_train, y_train, cv=cv, method="predict_proba",
)[:, 1]

fig, ax = plt.subplots(figsize=(6, 6))
for nom, proba in [("Modèle actuel", proba_oof), ("Modèle calibré", proba_cal_oof)]:
    freq_reelle, proba_moyenne = calibration_curve(y_tr, proba, n_bins=10, strategy="quantile")
    ax.plot(proba_moyenne, freq_reelle, marker="o", label=nom)
ax.plot([0, 1], [0, 1], "--", color="grey", label="Calibration parfaite")
ax.set_xlabel("Probabilité prédite")
ax.set_ylabel("Fréquence réelle de churn")
ax.set_title("Courbe de calibration (out-of-fold)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Score de Brier (plus bas = mieux) : actuel {brier_score_loss(y_tr, proba_oof):.4f} | calibré {brier_score_loss(y_tr, proba_cal_oof):.4f}")
print(f"ROC-AUC : actuel {roc_auc_score(y_tr, proba_oof):.4f} | calibré {roc_auc_score(y_tr, proba_cal_oof):.4f}")

courbes_cal = metriques_par_seuil(y_tr, proba_cal_oof)
theorique = COUT_FAUX_POSITIF / (COUT_FAUX_POSITIF + COUT_FAUX_NEGATIF)
print(f"\nSeuil optimal (critère coût) : non calibré {seuil_cout(courbes):.2f} | calibré {seuil_cout(courbes_cal):.2f}")
print(f"Seuil théorique pour des probabilités calibrées : {theorique:.2f}")

**Lecture :** si la courbe du modèle actuel s'écarte nettement de la diagonale, ses scores sont à
présenter comme un **score de risque** (et non comme une probabilité) ou bien il faut adopter la version
calibrée. La calibration ne change presque pas l'AUC (le classement des clients reste le même),
mais elle déplace le seuil optimal : un seuil choisi sur un modèle ne vaut que pour ce modèle.

## 7. Confirmation sur le jeu de test

Une seule évaluation, **sans retour en arrière** sur le choix du seuil. On compare le seuil actuel
au seuil retenu avec le modèle actuel (non calibré).

In [ ]:
modele = nouveau_pipeline().fit(X_train, y_train)
proba_test = modele.predict_proba(X_test)[:, 1]

comparaison = metriques_par_seuil(y_te, proba_test, [SEUIL_ACTUEL, SEUIL_RETENU])
comparaison.index = [f"actuel ({SEUIL_ACTUEL})", f"retenu ({SEUIL_RETENU})"]
display(comparaison[["precision", "rappel", "f1", "specificite", "part_marquee", "tp", "fp", "fn", "tn", "cout"]].round(3))

print(f"ROC-AUC test : {roc_auc_score(y_te, proba_test):.4f} | précision moyenne : {average_precision_score(y_te, proba_test):.4f}")

cout_actuel, cout_retenu = comparaison["cout"].iloc[0], comparaison["cout"].iloc[1]
print(f"Coût (hypothèses : FP = {COUT_FAUX_POSITIF}, FN = {COUT_FAUX_NEGATIF}) : "
      f"{cout_actuel:.0f} -> {cout_retenu:.0f} ({(cout_retenu - cout_actuel) / cout_actuel:+.1%})")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, seuil, titre in zip(axes, [SEUIL_ACTUEL, SEUIL_RETENU], ["Seuil actuel", "Seuil retenu"]):
    cm = confusion_matrix(y_te, (proba_test >= seuil).astype(int))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Reste", "Churn"], yticklabels=["Reste", "Churn"])
    ax.set_title(f"{titre} ({seuil})")
    ax.set_xlabel("Prédit")
    ax.set_ylabel("Réel")
plt.tight_layout()
plt.show()

## 8. Conclusion et mise en production

Si le seuil retenu diffère de 0,5 et que les résultats du test confirment ceux de la validation croisée :

1. **API (`api/main.py`)** : remplacer la constante `THRESHOLD = 0.5` par le seuil retenu. Mieux : le
   lire depuis un fichier de configuration ou un artefact enregistré à l'entraînement, pour ne pas le
   dupliquer à la main.
2. **Streamlit** : présenter le chiffre affiché comme un *score de risque* si les probabilités ne sont
   pas calibrées, ou adopter le modèle calibré (modification de `src/models/train.py`, puis nouveau seuil
   à déterminer sur les probabilités calibrées).
3. **Tests** : adapter les tests qui supposent un seuil de 0,5 et relancer `pytest`.
4. **Traçabilité** : enregistrer le seuil et le critère choisi dans MLflow (par exemple avec `log_param`).

> Les coûts utilisés sont des hypothèses. Dans un cas réel, ils doivent venir de l'équipe métier
> (coût d'une offre de rétention, valeur d'un client conservé).